# Soft-token loop test: Qwen → Gumbel-softmax → Qwen judge

Generator and judge are the same `Qwen2.5-0.5B-Instruct` (shared weights). Each generated
token is a Gumbel-softmax vector `y = softmax((logits/T + g)/λ)`; it is fed back to the
generator and to the judge as the embedding `y @ E`. `λ = None` means hard Gumbel-max
sampling (one-hot `y`). The judge never sees the instruction, only the response.

Sentiment readout: `s = logsumexp(logit[positive, Positive]) − logsumexp(logit[negative, Negative])`
at the judge's first answer token.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# JUDGE = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cpu"  # the local GPU has 1 GB, too small for the fp32 model
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device).eval()
# judge_model = AutoModelForCausalLM.from_pretrained(JUDGE, dtype=torch.float32).to(device).eval()
judge_model = model
for p in model.parameters():
    p.requires_grad_(False)
E = model.get_input_embeddings().weight          # (V, d), V = logit dimension
E_J = judge_model.get_input_embeddings().weight
STOP = {tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}
POS = [tok.encode(w)[0] for w in ("positive", "Positive")]
NEG = [tok.encode(w)[0] for w in ("negative", "Negative")]
print(device, E.shape)

cpu torch.Size([151936, 896])


In [ ]:
BASE = "Write one short sentence about the public transport system."
INSTRUCTIONS = [
    "Be extremely enthusiastic and positive.",
    # "Be strictly neutral and impersonal; use no positive or negative language.",
    "Be extremely critical and negative.",
]

def chat_ids(user_text):
    text = tok.apply_chat_template([{"role": "user", "content": user_text}],
                                   tokenize=False, add_generation_prompt=True)
    return torch.tensor(tok.encode(text), device=device)

# Judge template, split around the response slot.
JUDGE_PROMPT = ("What is the BERT sentiment of the following text? Answer with one word: "
         "positive or negative.\n\nText: {R}")
_t = tok.apply_chat_template([{"role": "user", "content": JUDGE_PROMPT}],
                             tokenize=False, add_generation_prompt=True)
_pre, _suf = _t.split("{R}")
PRE = torch.tensor(tok.encode(_pre), device=device)
SUF = torch.tensor(tok.encode(_suf), device=device)

In [ ]:
def generate(prompt_emb, lam=None, seed=0, T=1.0, max_new=32, length=None):
    """Gumbel decoding from prompt embeddings (L0, d).
    Returns soft tokens Y (L, V) and the Gumbel-max ids (argmax of each y).
    `length` forces exactly that many tokens, ignoring stop tokens: the stop rule
    makes the map discontinuous, so perturbed runs must not change the length."""
    gen = torch.Generator(device=device).manual_seed(seed)
    out = model(inputs_embeds=prompt_emb[None], use_cache=True)
    past, logits = out.past_key_values, out.logits[0, -1]
    Y, ids = [], []
    for _ in range(length or max_new):
        u = torch.rand(logits.shape, generator=gen, device=device).clamp_min(1e-20)
        z = logits / T - torch.log(-torch.log(u))            # perturbed logits
        i = int(z.argmax())
        if length is None and i in STOP:
            break
        y = F.one_hot(torch.tensor(i, device=device), z.numel()).float() if lam is None \
            else torch.softmax(z / lam, dim=-1)
        Y.append(y); ids.append(i)
        out = model(inputs_embeds=(y @ E)[None, None], past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[0, -1]
    return torch.stack(Y), ids

def judge(Y):
    emb = torch.cat([E_J[PRE], Y @ E_J, E_J[SUF]])
    lg = judge_model(inputs_embeds=emb[None]).logits[0, -1]
    return torch.logsumexp(lg[POS], 0) - torch.logsumexp(lg[NEG], 0)

def full_prompt_emb(instruction):
    return E[chat_ids(f"{BASE} {instruction}")]

## 1. Hard vs soft scores

For each prompt, seed and `λ`: the text read off the argmax tokens, the judge's score on
the soft tokens `s_soft`, and on the one-hot version of the same argmax tokens `s_hard`.
Same seed ⇒ same Gumbel noise across `λ`; the texts can still differ, because soft
tokens fed back into the generator change its later logits.

In [ ]:
LAMS = [None, 0.1, 0.2, 0.5]
rows = []
with torch.no_grad():
    for ins in INSTRUCTIONS:
        P = full_prompt_emb(ins)
        for seed in range(5):
            for lam in LAMS:
                Y, ids = generate(P, lam=lam, seed=seed)
                Yh = F.one_hot(torch.tensor(ids, device=device), Y.shape[1]).float()
                rows.append(dict(instruction=ins[:25], seed=seed, lam=lam,
                                 s_soft=judge(Y).item(), s_hard=judge(Yh).item(),
                                 peak=Y.max(1).values.mean().item(),
                                 text=tok.decode(ids)))
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
df

,instruction,seed,lam,s_soft,s_hard,peak,text
0,Be extremely enthusiastic,0,NaN,1.841509,1.841509,1.000000,"Public transport makes getting around much easier and more efficient, giving us freedom to explore our city and enjo..."
1,Be extremely enthusiastic,0,0.1,3.973448,3.883064,0.907794,Public transport systems are treasures that not only facilitate travel but also promote community activities and eco...
2,Be extremely enthusiastic,0,0.2,4.116869,3.536884,0.856829,Public transit systems are treasures that not only facilitate daily but also promote community activities and econom...
3,Be extremely enthusiastic,0,0.5,3.938595,3.192942,0.782664,Public transit systems are treasures that not only facilitate daily freedom but also encaps activities like families...
4,Be extremely enthusiastic,1,NaN,5.115742,5.115742,1.000000,"I'm so glad to see my travels optimally organized, and I'm absolutely thrilled to contribute to making public transp..."
5,Be extremely enthusiastic,1,0.1,5.296791,5.198168,0.953446,"I'm so glad to see my travels optimally scheduled, and I'm absolutely thrilled to contribute to making public transp..."
6,Be extremely enthusiastic,1,0.2,4.768408,4.941877,0.934775,I'm so glad to see my travels optimistically linked to the efficient public transport system!
7,Be extremely enthusiastic,1,0.5,4.094027,3.983715,0.753819,"I'm so glad to see my friends using the Public Transport twice within this sector, it's actually quite great!"
8,Be extremely enthusiastic,2,NaN,3.966146,3.966146,1.000000,"Amazingly, public transportation makes getting around the city so easy and efficient you never even notice it's happ..."
9,Be extremely enthusiastic,2,0.1,0.845959,1.317492,0.936759,"Amazing everyone, immerse yourself in the vibrant tapestry of public transportation that made possible through innov..."


In [ ]:
# Agreement between soft and hard scores, and mean peak mass of y, per lambda
df.assign(gap=(df.s_soft - df.s_hard).abs()).groupby("lam", dropna=False)[["gap", "peak"]].mean()

,gap,peak
lam,,
0.1,0.156605,0.935366
0.2,0.248612,0.895260
0.5,0.230662,0.773411
NaN,0.000000,1.000000


In [ ]:
df.groupby(["instruction", "lam"], dropna=False)[["s_soft", "s_hard"]].mean()

s_soft    s_hard
instruction               lam                    
Be extremely critical and 0.1 -3.445428 -3.463568
                          0.2 -2.707297 -2.701479
                          0.5 -3.948284 -4.070145
                          NaN -3.712981 -3.712981
Be extremely enthusiastic 0.1  3.326182  3.243224
                          0.2  3.721449  3.727311
                          0.5  2.858506  2.668955
                          NaN  3.528462  3.528462

In [ ]:
del(df)

## 2. Gradient through the loop

Gradient of `s` with respect to the prompt embeddings. With hard tokens there is no path
from the prompt to `s` (the judge sees only one-hot vectors), so the gradient is zero. With
soft tokens it is not. The finite-difference check along a random direction `δ` (same seed
on both sides) tests that autograd matches the relaxed system's actual sensitivity.

In [ ]:
P0 = full_prompt_emb(INSTRUCTIONS[1])
torch.manual_seed(0)
delta = torch.randn_like(P0); delta /= delta.norm()
h = 1e-4
for lam in LAMS:
    P = P0.clone().requires_grad_(True)
    Y, _ = generate(P, lam=lam, seed=0)
    L = Y.shape[0]  # pin the perturbed runs to the base length
    s = judge(Y)
    if not s.requires_grad:
        print(f"lam={lam}: s={s.item():+.3f}  no gradient path (hard tokens)")
        continue
    s.backward()
    with torch.no_grad():
        response = generate(P0 + h * delta, lam=lam, seed=0, length=L)[0]
        print(response.shape)
        fd = (judge(generate(P0 + h * delta, lam=lam, seed=0, length=L)[0])
              - judge(generate(P0 - h * delta, lam=lam, seed=0, length=L)[0])) / (2 * h)
    print(f"lam={lam}: s={s.item():+.3f}  |grad|={P.grad.norm():.3f}  "
          f"grad·δ={(P.grad * delta).sum():+.4f}  finite diff={fd.item():+.4f}")

lam=None: s=-4.225  no gradient path (hard tokens)
torch.Size([32, 151936])
lam=0.1: s=-3.071  |grad|=327.606  grad·δ=+1.3266  finite diff=+1.3828
torch.Size([32, 151936])
lam=0.2: s=-3.154  |grad|=101.377  grad·δ=-0.5905  finite diff=-0.6485
torch.Size([20, 151936])
lam=0.5: s=-3.788  |grad|=368.615  grad·δ=+0.3391  finite diff=-0.0095
